# Zeta Phase Map

Time-colored phase trajectories for selected countries.

- x-axis: lagged `zeta`
- y-axis: current `zeta`


In [ ]:
from pathlib import Path
import os
import sys
import importlib

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from io_networks.config import load_config
import io_networks.viz_zeta as viz_zeta
importlib.reload(viz_zeta)

load_zeta_data = viz_zeta.load_zeta_data
select_phase_map_countries = viz_zeta.select_phase_map_countries
plot_zeta_phase_map = viz_zeta.plot_zeta_phase_map

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np


In [ ]:
cfg = load_config('config/default.yaml')
variant = None  # None -> inferred from config
only_ok = True
# Truncated Oranges: avoids near-white low end while keeping early years lighter.
base_cmap = plt.get_cmap('Oranges')
cmap = LinearSegmentedColormap.from_list(
    'Oranges_trunc',
    base_cmap(np.linspace(0.22, 1.0, 256))
)
year_step = 5  # keep one point every N years


In [ ]:
df = load_zeta_data(cfg=cfg, variant=variant)
print('rows:', len(df))
print('years:', int(df['year'].min()), 'to', int(df['year'].max()))
if 'status' in df.columns:
    print('status counts:', df['status'].value_counts(dropna=False).to_dict())


In [ ]:
auto_phase_countries = select_phase_map_countries(df, only_ok=only_ok)
print('auto placeholders:', auto_phase_countries)

# Override manually if needed
phase_countries = ['CHN', 'USA', 'BRA', 'IND', 'MEX', 'VNM']


In [ ]:
base_year = int(df['year'].min())
df_phase = df[(df['year'] - base_year) % year_step == 0].copy()
print('phase rows:', len(df_phase), 'from original:', len(df))

fig, ax = plt.subplots(figsize=(12, 8))
plot_zeta_phase_map(
    df_phase,
    countries=phase_countries,
    only_ok=only_ok,
    title=r'Phase map: $\zeta_t$ vs $\zeta_{t-1}$ (time-colored)',
    cmap=cmap,
    lw=2.6,
    point_size=28,
    show_end_labels=True,
    end_label_dx_pts=12,
    end_label_dy_pts=9,
    end_label_bbox_alpha=0.85,
    show_background=True,
    bg_color='0.55',
    bg_alpha=0.12,
    bg_lw=0.9,
    show_flow_arrows=True,
    flow_arrow_step=2,
    flow_arrow_lw=2,
    flow_arrow_scale=25,
    ax=ax,
)
plt.tight_layout()
plt.show()
